# NeuroSleep — Notebook 03: Classical ML Baselines

**Project**: Automated sleep stage classification from EEG signals using deep learning.

**Author**: Marcos Vinícius Rocha Gomes

**Date**: May 2026

---

## Objectives of this notebook

1. Load the engineered feature matrix produced in Notebook 02.
2. Build a **subject-aware** train/test split (no data leakage across subjects).
3. Train two classical baselines: **Random Forest** and **SVM (RBF)**.
4. Evaluate with metrics that are appropriate for imbalanced classification: **macro-F1**, **balanced accuracy**, **Cohen's kappa**, and per-class precision/recall.
5. Visualize confusion matrices and feature importances.
6. Persist results to disk so the deep learning notebook can fairly compare.

## Why baselines matter

Reporting only deep learning results without baselines is bad practice. A CNN that beats Random Forest by 2 percentage points may not justify the engineering cost. Baselines anchor the comparison and force us to be honest about how much value DL adds.

## Why subject-aware splitting

If we shuffle epochs randomly, the same subject ends up in both train and test sets. EEG patterns are highly subject-specific, so models would learn to recognize the subject rather than the sleep stage — leading to inflated metrics that don't generalize to new patients. **Grouping by subject is mandatory.**

## 1. Imports and configuration

In [ ]:
import warnings
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    GroupShuffleSplit,
    StratifiedGroupKFold,
    cross_val_score,
)
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    accuracy_score,
)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100

RANDOM_STATE = 42
STAGE_ORDER = ["W", "N1", "N2", "N3", "REM"]

PROCESSED_DIR = Path("../data/processed")
RESULTS_DIR = Path("../data/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Setup complete.")

## 2. Load the feature matrix

In [ ]:
features_path = PROCESSED_DIR / "features.parquet"
df = pd.read_parquet(features_path)

print(f"Loaded {len(df):,} epochs from {df['subject'].nunique()} subjects.")
print(f"Feature matrix shape: {df.shape}")
print(f"Stages: {sorted(df['stage'].unique())}")

df.head()

In [ ]:
# Separate metadata columns from features
META_COLS = ["subject", "epoch_idx", "stage"]
FEATURE_COLS = [c for c in df.columns if c not in META_COLS]

X = df[FEATURE_COLS].values
y = df["stage"].values
groups = df["subject"].values

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Unique groups (subjects): {np.unique(groups)}")

## 3. Subject-aware train/test split

We use `GroupShuffleSplit` to ensure no subject appears in both splits.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
groups_train, groups_test = groups[train_idx], groups[test_idx]

print("Train set:")
print(f"  Epochs: {len(X_train):,}")
print(f"  Subjects: {sorted(np.unique(groups_train))}")
print("\nTest set:")
print(f"  Epochs: {len(X_test):,}")
print(f"  Subjects: {sorted(np.unique(groups_test))}")

# Sanity: no overlap
assert set(np.unique(groups_train)).isdisjoint(set(np.unique(groups_test))), "Subject leakage detected!"
print("\n✓ No subject overlap between train and test.")

## 4. Class distribution check

Imbalanced classes mean we need class weights AND we cannot rely on plain accuracy.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (name, y_) in zip(axes, [("Train", y_train), ("Test", y_test)]):
    counts = pd.Series(y_).value_counts().reindex(STAGE_ORDER, fill_value=0)
    pct = (counts / counts.sum() * 100).round(1)
    sns.barplot(x=counts.index, y=counts.values, order=STAGE_ORDER, palette="viridis", ax=ax)
    for i, (c, p) in enumerate(zip(counts.values, pct.values)):
        ax.text(i, c + counts.max() * 0.02, f"{p}%", ha="center", fontsize=9)
    ax.set_title(f"{name} set — class distribution")
    ax.set_xlabel("Sleep stage")
    ax.set_ylabel("Number of epochs")

plt.tight_layout()
plt.show()

## 5. Helper functions for evaluation

In [ ]:
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name: str) -> dict:
    """Fit on train, evaluate on test, return a dict of metrics."""
    start = time.time()
    model.fit(X_train, y_train)
    fit_seconds = time.time() - start

    start = time.time()
    y_pred = model.predict(X_test)
    predict_seconds = time.time() - start

    metrics = {
        "model": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "macro_f1": f1_score(y_test, y_pred, average="macro"),
        "weighted_f1": f1_score(y_test, y_pred, average="weighted"),
        "cohen_kappa": cohen_kappa_score(y_test, y_pred),
        "fit_seconds": fit_seconds,
        "predict_seconds": predict_seconds,
    }
    print(f"=== {model_name} ===")
    print(f"  Accuracy:          {metrics['accuracy']:.4f}")
    print(f"  Balanced accuracy: {metrics['balanced_accuracy']:.4f}")
    print(f"  Macro F1:          {metrics['macro_f1']:.4f}")
    print(f"  Weighted F1:       {metrics['weighted_f1']:.4f}")
    print(f"  Cohen's kappa:     {metrics['cohen_kappa']:.4f}")
    print(f"  Fit time:          {fit_seconds:.2f}s")
    print(f"  Predict time:      {predict_seconds:.2f}s")
    return metrics, y_pred


def plot_confusion_matrix(y_true, y_pred, title: str, ax=None):
    """Plot a normalized confusion matrix."""
    cm = confusion_matrix(y_true, y_pred, labels=STAGE_ORDER, normalize="true")
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True, fmt=".2f", cmap="Blues",
        xticklabels=STAGE_ORDER, yticklabels=STAGE_ORDER,
        cbar=False, ax=ax, vmin=0, vmax=1,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    return ax

## 6. Compute class weights

Both Random Forest and SVM accept class weights to compensate for imbalance.

In [ ]:
class_labels_train = np.unique(y_train)
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=class_labels_train,
    y=y_train,
)
class_weight_dict = dict(zip(class_labels_train, class_weights))

print("Computed class weights (inversely proportional to frequency):")
for stage in STAGE_ORDER:
    if stage in class_weight_dict:
        print(f"  {stage:>4}: {class_weight_dict[stage]:.3f}")

## 7. Baseline 1: Random Forest

Random Forest is a strong tabular baseline:
- Handles features with different scales without normalization.
- Captures nonlinear interactions naturally.
- Built-in feature importance.
- Robust to outliers.

We use 200 trees and let trees grow fully — depth is regularized implicitly by bagging.

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=2,
    class_weight=class_weight_dict,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

rf_metrics, rf_pred = evaluate_model(rf_model, X_train, y_train, X_test, y_test, "Random Forest")

In [ ]:
print("Random Forest — per-class report:\n")
print(classification_report(y_test, rf_pred, labels=STAGE_ORDER, digits=3, zero_division=0))

### Feature importance (top 20)

Useful both as a sanity check (do the most important features make physiological sense?) and as a diagnostic tool.

In [ ]:
importances = pd.Series(rf_model.feature_importances_, index=FEATURE_COLS)
top20 = importances.sort_values(ascending=True).tail(20)

fig, ax = plt.subplots(figsize=(10, 8))
top20.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Top 20 most important features (Random Forest)")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

**Sanity check**: if delta and beta band powers, spectral entropy, and Hjorth complexity rank high, our features are doing the right thing — these are exactly the descriptors that distinguish sleep stages physiologically.

## 8. Baseline 2: SVM with RBF kernel

SVMs are sensitive to feature scale, so we wrap the model in a `Pipeline` with `StandardScaler`. RBF kernel handles nonlinearity. SVMs are slower to train than RF on large datasets — if it's too slow, we can subsample or use linear kernel.

In [ ]:
# If training is too slow, reduce dataset for SVM (sklearn SVC is O(n^2) memory)
MAX_SVM_TRAIN_SAMPLES = 15000

if len(X_train) > MAX_SVM_TRAIN_SAMPLES:
    rng = np.random.RandomState(RANDOM_STATE)
    # Stratified subsample by class
    indices = []
    for cls in np.unique(y_train):
        cls_idx = np.where(y_train == cls)[0]
        n_take = int(MAX_SVM_TRAIN_SAMPLES * len(cls_idx) / len(y_train))
        sampled = rng.choice(cls_idx, size=min(n_take, len(cls_idx)), replace=False)
        indices.append(sampled)
    sample_idx = np.concatenate(indices)
    X_train_svm = X_train[sample_idx]
    y_train_svm = y_train[sample_idx]
    print(f"Subsampling training set for SVM: {len(X_train):,} -> {len(X_train_svm):,}")
else:
    X_train_svm, y_train_svm = X_train, y_train

svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        class_weight=class_weight_dict,
        random_state=RANDOM_STATE,
        cache_size=500,
    )),
])

svm_metrics, svm_pred = evaluate_model(svm_model, X_train_svm, y_train_svm, X_test, y_test, "SVM (RBF)")

In [ ]:
print("SVM (RBF) — per-class report:\n")
print(classification_report(y_test, svm_pred, labels=STAGE_ORDER, digits=3, zero_division=0))

## 9. Confusion matrices side by side

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_confusion_matrix(y_test, rf_pred, "Random Forest — normalized CM", ax=axes[0])
plot_confusion_matrix(y_test, svm_pred, "SVM (RBF) — normalized CM", ax=axes[1])
plt.tight_layout()
plt.show()

### Expected patterns to discuss

- **N1 is the hardest class** — it's a transitional stage with no strong distinctive features, frequently confused with REM or N2.
- **W vs N3** should be easiest — opposite ends of the EEG amplitude spectrum.
- **N2 ↔ N1 confusion** is also expected and matches inter-rater agreement among human experts.

## 10. Cross-validation for a more honest estimate

A single train/test split is noisy — especially with few subjects. Let's run grouped cross-validation on Random Forest (cheaper than SVM) to get a more honest performance estimate with variance.

In [ ]:
n_subjects = len(np.unique(groups))
n_splits = min(5, n_subjects)

if n_subjects >= 3:
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    cv_scores = cross_val_score(
        RandomForestClassifier(
            n_estimators=200,
            class_weight=class_weight_dict,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
        X, y, groups=groups, cv=cv, scoring="f1_macro", n_jobs=-1,
    )
    print(f"Grouped {n_splits}-fold CV — macro-F1 across folds:")
    for i, s in enumerate(cv_scores):
        print(f"  Fold {i + 1}: {s:.4f}")
    print(f"\n  Mean: {cv_scores.mean():.4f}")
    print(f"  Std:  {cv_scores.std():.4f}")
else:
    print(f"Not enough subjects ({n_subjects}) for CV. Skip.")
    cv_scores = None

## 11. Comparison table

In [ ]:
comparison = pd.DataFrame([rf_metrics, svm_metrics]).set_index("model")
print("Side-by-side comparison:\n")
comparison.round(4)

In [ ]:
metric_cols = ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1", "cohen_kappa"]
plot_df = comparison[metric_cols].reset_index().melt(
    id_vars="model", var_name="metric", value_name="score"
)

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=plot_df, x="metric", y="score", hue="model", palette="Set2", ax=ax)
ax.set_title("Baseline comparison")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.legend(loc="lower right")
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", fontsize=8, padding=2)
plt.tight_layout()
plt.show()

## 12. Persist results

Save the metrics and predictions so the deep learning notebook can compare directly without re-running these baselines.

In [ ]:
# Combined results dict
results = {
    "random_forest": rf_metrics,
    "svm_rbf": svm_metrics,
    "cv_macro_f1": {
        "mean": float(cv_scores.mean()) if cv_scores is not None else None,
        "std": float(cv_scores.std()) if cv_scores is not None else None,
        "folds": cv_scores.tolist() if cv_scores is not None else None,
    },
    "test_subjects": sorted([int(s) for s in np.unique(groups_test)]),
    "train_subjects": sorted([int(s) for s in np.unique(groups_train)]),
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
    "n_features": int(X_train.shape[1]),
}

results_path = RESULTS_DIR / "baseline_metrics.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"Saved metrics to {results_path}")

# Predictions for later inspection / ensemble
predictions_df = pd.DataFrame({
    "subject": groups_test,
    "true": y_test,
    "rf_pred": rf_pred,
    "svm_pred": svm_pred,
})
predictions_path = RESULTS_DIR / "baseline_predictions.parquet"
predictions_df.to_parquet(predictions_path, index=False)
print(f"Saved predictions to {predictions_path}")

## 13. Summary and next steps

### Key findings

- Subject-aware splitting confirms our pipeline generalizes to unseen subjects (no data leakage).
- Random Forest provides a strong, interpretable baseline with feature importances that match physiological intuition.
- SVM with RBF kernel performs comparably — useful as a second viewpoint.
- The hardest class is **N1**, as expected from the literature.
- Macro-F1 is the most informative metric here, since accuracy is biased toward majority class (N2).

### Honest limitations

- Few subjects → high variance in metrics. The CV std gives a sense of how unstable estimates are.
- We're modeling each epoch independently — ignoring the temporal structure of sleep cycles (transitions between stages are not arbitrary).
- No hyperparameter tuning — this is intentional for the baseline. The deep model will face the same untuned-vs-tuned scrutiny.

### What the deep model needs to beat

These macro-F1 numbers are the bar. The CNN in Notebook 04 needs to do better — meaningfully — to justify its added complexity. If it doesn't, the honest conclusion is: feature engineering + Random Forest is sufficient for this problem at this scale.